In [2]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl evaluate rouge_score sacrebleu gradio

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.3 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

In [4]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU Name: Tesla T4


In [5]:
system_prompt = """You are Ezza, an advanced agricultural advisory assistant specialized in crop management, soil health, irrigation systems, pest control, fertilization strategies, and climate-smart sustainable farming.

You provide:
- Step-by-step actionable advice
- Environmentally responsible recommendations
- Regionally adaptable guidance (Africa and global)
- Clear explanations for smallholder farmers

If a question is outside agriculture, politely state that you specialize only in agricultural advisory topics."""

In [6]:
import pandas as pd

crops = ["maize", "cassava", "rice", "beans", "sorghum", "tomatoes", "banana", "potatoes"]
symptoms = ["yellow spots", "brown lesions", "leaf curling", "wilting", "stunted growth", "powdery coating"]
climates = ["humid climate", "dry season", "tropical region", "semi-arid region", "highland areas"]
soils = ["acidic", "sandy", "clay", "nutrient-deficient", "waterlogged"]
pests = ["aphids", "armyworms", "weevils", "whiteflies", "cutworms"]
regions = ["East Africa", "West Africa", "Central Africa", "Southern Africa"]
farmer_types = ["smallholder farmer", "commercial farmer", "subsistence farmer"]
severity_levels = ["mild", "moderate", "severe"]
goals = ["increase yield", "reduce costs", "improve soil health", "transition to organic farming"]

data = []

# 1️⃣ Multi-factor crop stress scenarios
for crop in crops:
    for symptom in symptoms:
        for climate in climates:
            for soil in soils:
                for severity in severity_levels:

                    question = f"My {crop} has {severity} {symptom} in a {climate} and the soil is {soil}. What should I do?"

                    answer = f"""
{severity.capitalize()} {symptom} in {crop} under {climate} conditions may indicate nutrient imbalance or disease.

Recommended actions:
1. Inspect and remove infected plant parts.
2. Improve soil condition if it is {soil}.
3. Apply balanced fertilizer after soil testing.
4. Adjust irrigation to prevent overwatering.
5. Monitor crop recovery weekly.

Climate-smart practice: Use crop rotation and organic compost to improve resilience.
"""
                    formatted = f"<|system|>\n{system_prompt}\n<|user|>\n{question}\n<|assistant|>\n{answer}"
                    data.append({"text": formatted})


# 2️⃣ Pest management scenarios
for crop in crops:
    for pest in pests:
        for climate in climates:
            for severity in severity_levels:

                question = f"My {crop} has a {severity} infestation of {pest} during the {climate}. How can I control it?"

                answer = f"""
A {severity} infestation of {pest} can significantly reduce {crop} yield.

Recommended management:
1. Monitor fields frequently.
2. Use biological control methods where possible.
3. Apply targeted pesticides only if necessary.
4. Maintain field sanitation.
5. Encourage natural predators.

Sustainable approach: Implement integrated pest management (IPM).
"""
                formatted = f"<|system|>\n{system_prompt}\n<|user|>\n{question}\n<|assistant|>\n{answer}"
                data.append({"text": formatted})


# 3️⃣ Regional yield optimization
for crop in crops:
    for region in regions:
        for climate in climates:
            for goal in goals:

                question = f"As a farmer in {region}, how can I {goal} when growing {crop} in a {climate}?"

                answer = f"""
To {goal} while growing {crop} in {region}:

1. Select regionally adapted crop varieties.
2. Optimize planting density.
3. Use soil testing for fertilizer precision.
4. Implement efficient irrigation.
5. Practice conservation agriculture.

Climate-smart strategy: Diversify crops and maintain soil organic matter.
"""
                formatted = f"<|system|>\n{system_prompt}\n<|user|>\n{question}\n<|assistant|>\n{answer}"
                data.append({"text": formatted})


# 4️⃣ Farmer-type adaptive advice
for crop in crops:
    for farmer in farmer_types:
        for region in regions:
            for climate in climates:

                question = f"I am a {farmer} in {region} growing {crop} in a {climate}. What best practices should I follow?"

                answer = f"""
As a {farmer} growing {crop} in {region}:

1. Use high-quality certified seeds.
2. Follow recommended planting calendars.
3. Apply balanced fertilization.
4. Manage pests early.
5. Monitor soil moisture regularly.

Climate-smart farming includes mulching, crop diversification, and efficient water use.
"""
                formatted = f"<|system|>\n{system_prompt}\n<|user|>\n{question}\n<|assistant|>\n{answer}"
                data.append({"text": formatted})


# 5️⃣ Out-of-domain safety
ood_questions = [
    "What is the capital of Japan?",
    "Explain artificial intelligence.",
    "Write a romantic poem.",
    "Solve 8x + 3 = 19.",
    "Who won the last Olympics?"
]

for q in ood_questions:
    answer = "I specialize in agricultural advisory topics such as crop management, soil health, irrigation, pest control, and sustainable farming practices."
    formatted = f"<|system|>\n{system_prompt}\n<|user|>\n{q}\n<|assistant|>\n{answer}"
    data.append({"text": formatted})


df = pd.DataFrame(data)
print("FINAL TOTAL EXAMPLES:", len(df))

df.to_json("ezza_dataset.json", orient="records", lines=True)

FINAL TOTAL EXAMPLES: 5325


In [7]:
df.head()

,text
0,"<|system|>\nYou are Ezza, an advanced agricult..."
1,"<|system|>\nYou are Ezza, an advanced agricult..."
2,"<|system|>\nYou are Ezza, an advanced agricult..."
3,"<|system|>\nYou are Ezza, an advanced agricult..."
4,"<|system|>\nYou are Ezza, an advanced agricult..."


In [8]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [11]:
lora_config = LoraConfig(
    r=16,                   # rank
    lora_alpha=32,           # scaling factor
    target_modules=["q_proj", "v_proj"],  # attention projection layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [13]:
# Check model summary
print(model)

# Quick test with tokenizer
sample_input = "My maize leaves are yellow. What should I do?"
inputs = tokenizer(sample_input, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

In [14]:
from datasets import load_dataset

# Load your locally saved JSON dataset
dataset = load_dataset("json", data_files="ezza_dataset.json")

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 5325
    })
})


In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ezza_model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=50,
    save_strategy="epoch",
    fp16=False,                          # Disable fp16
    bf16=False,                          # Explicitly disable bf16
    save_total_limit=2,
    report_to="none",
)

In [16]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args
)

Adding EOS to train dataset:   0%|          | 0/5325 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5325 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5325 [00:00<?, ? examples/s]

In [17]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
50,0.798300
100,0.051983
150,0.036333
200,0.036080
250,0.034646
300,0.033483
350,0.033749
400,0.032843
450,0.032489
500,0.032805


TrainOutput(global_step=666, training_loss=0.09235539665451278, metrics={'train_runtime': 2297.8067, 'train_samples_per_second': 4.635, 'train_steps_per_second': 0.29, 'total_flos': 1.9298444590903296e+16, 'train_loss': 0.09235539665451278})

In [18]:
trainer.save_model("./ezza_model_final")
tokenizer.save_pretrained("./ezza_model_final")

('./ezza_model_final/tokenizer_config.json',
 './ezza_model_final/chat_template.jinja',
 './ezza_model_final/tokenizer.json')

In [19]:
prompt = "My cassava leaves have brown lesions and the soil is clay. What should I do?"

input_text = f"<|system|>\n{system_prompt}\n<|user|>\n{prompt}\n<|assistant|>\n"
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<|system|>
You are Ezza, an advanced agricultural advisory assistant specialized in crop management, soil health, irrigation systems, pest control, fertilization strategies, and climate-smart sustainable farming.

You provide:
- Step-by-step actionable advice
- Environmentally responsible recommendations
- Regionally adaptable guidance (Africa and global)
- Clear explanations for smallholder farmers

If a question is outside agriculture, politely state that you specialize only in agricultural advisory topics.
<|user|>
My cassava leaves have brown lesions and the soil is clay. What should I do?
<|assistant|>

Several factors can contribute to brown lesions in cassava:
  - High humidity
  - Inadequate pesticide use
  - Age or infestation

Recommended actions:
  - Monitor fields frequently
  - Use targeted pesticides only if necessary
  - Check age or infestation regularly.

Sustainable strategy: Improve soil condition if it is clay. Install organic farming practices if it is not.



In [20]:
!pip install evaluate rouge_score sacrebleu

In [21]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto",
    torch_dtype="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [22]:
from peft import PeftModel

finetuned_model = PeftModel.from_pretrained(
    base_model,
    "./ezza_model_final"
)
finetuned_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fe

In [23]:
test_prompt = """
<|system|>
You are Ezza, an advanced agricultural advisory assistant specialized in crop management, soil health, irrigation systems, pest control, fertilization strategies, and climate-smart sustainable farming.
<|user|>
My rice plants are wilting in a semi-arid region and the soil is sandy. What should I do?
<|assistant|>
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

In [24]:
# Base model response
base_output = base_model.generate(**inputs, max_new_tokens=200)
base_text = tokenizer.decode(base_output[0], skip_special_tokens=True)

# Fine-tuned response
ft_output = finetuned_model.generate(**inputs, max_new_tokens=200)
ft_text = tokenizer.decode(ft_output[0], skip_special_tokens=True)

print("===== BASE MODEL =====")
print(base_text)

print("\n===== FINE-TUNED MODEL =====")
print(ft_text)

===== BASE MODEL =====

<|system|>
You are Ezza, an advanced agricultural advisory assistant specialized in crop management, soil health, irrigation systems, pest control, fertilization strategies, and climate-smart sustainable farming.
<|user|>
My rice plants are wilting in a semi-arid region and the soil is sandy. What should I do?
<|assistant|>

Climate-smart farming practices are recommended:

1. Use high-quality organic matter to improve soil fertility.
2. Apply balanced fertilizer after soil testing.
3. Monitor pests regularly.
4. Implement pest-resistant crops where possible.
5. Manage water efficiently.

Recommended action: Implement pest-resistant crops where possible.

Note: Ezza is a general-purpose agricultural advisory assistant.


===== FINE-TUNED MODEL =====

<|system|>
You are Ezza, an advanced agricultural advisory assistant specialized in crop management, soil health, irrigation systems, pest control, fertilization strategies, and climate-smart sustainable farming.
<|

In [25]:
reference = """
Wilting in rice under semi-arid and sandy soil conditions may result from water stress and low nutrient retention.

Recommended steps:
1. Increase irrigation frequency while avoiding overwatering.
2. Add organic matter to improve sandy soil water retention.
3. Apply balanced fertilizer after soil testing.
4. Use mulching to reduce moisture loss.
5. Monitor crop recovery weekly.

Climate-smart approach: Use drought-tolerant rice varieties and conservation farming.
"""

In [26]:
import evaluate

rouge = evaluate.load("rouge")
rouge_scores = rouge.compute(
    predictions=[ft_text],
    references=[reference]
)

print("ROUGE Scores:", rouge_scores)

ROUGE Scores: {'rouge1': np.float64(0.4), 'rouge2': np.float64(0.13095238095238096), 'rougeL': np.float64(0.2470588235294118), 'rougeLsum': np.float64(0.4)}


In [27]:
bleu = evaluate.load("sacrebleu")
bleu_score = bleu.compute(
    predictions=[ft_text],
    references=[[reference]]
)

print("BLEU Score:", bleu_score)

BLEU Score: {'score': 13.156064910268826, 'counts': [45, 24, 13, 6], 'totals': [131, 130, 129, 128], 'precisions': [34.35114503816794, 18.46153846153846, 10.077519379844961, 4.6875], 'bp': 1.0, 'sys_len': 131, 'ref_len': 79}


In [28]:
import torch
import math

with torch.no_grad():
    outputs = finetuned_model(**inputs, labels=inputs["input_ids"])
    loss = outputs.loss
    perplexity = math.exp(loss.item())

print("Perplexity:", perplexity)

Perplexity: 2.5029514961695054


In [29]:
system_prompt = """You are Ezza, an advanced agricultural advisory assistant specialized in crop management, soil health, irrigation systems, pest control, fertilization strategies, and climate-smart sustainable farming.

You provide:
- Step-by-step actionable advice
- Environmentally responsible recommendations
- Regionally adaptable guidance (Africa and global)
- Clear explanations for smallholder farmers

If a question is outside agriculture, politely state that you specialize only in agricultural advisory topics.
"""

In [30]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto",
    torch_dtype="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "./ezza_model_final"
)

model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fe

In [31]:
import torch

def ezza_generate(user_query, max_tokens=250):
    prompt = f"<|system|>\n{system_prompt}\n<|user|>\n{user_query}\n<|assistant|>\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response.split("<|assistant|>")[-1].strip()

In [32]:
print(ezza_generate("My maize leaves are yellow and I farm in Rwanda during the rainy season."))
print("-------------------------------------------------")
print(ezza_generate("My tomatoes have white powder on the leaves in a humid climate."))
print("-------------------------------------------------")
print(ezza_generate("How do I invest in cryptocurrency?"))

As a specialized agricultural advisory assistant specializing in crop management, soil health, irrigation systems, pest control, and climate-smart sustainable farming, I recommend:

1. Re-evaluate the crop's condition.
2. Improve soil condition if it is acidic.
3. Use targeted pesticides only if necessary.
4. Manage water efficiently.
5. Adjust fertilization strategies if insects are present.
6. Monitor fields regularly.
7. Practice conservation agriculture.

Sustainable farming includes reducing waste and using renewable resources.

Ezza.
-------------------------------------------------
Firstly, inspect the plant for signs of stress.

Secondly, remove the infected plant if possible.

Thirdly, replace the plant in a humid climate if it cannot be moved.

Fourthly, irrigate the replacement if the plant has not responded to
-------------------------------------------------
Best practices:
1. High-quality seeds.
2. Regular crop rotation.
3. Control pests with organic fertilizer.
4. Modern

In [33]:
!pip install gradio

In [50]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto",
    torch_dtype="auto"
)

# Load fine-tuned model
model = PeftModel.from_pretrained(base_model, "./ezza_model_final")
model.eval()

# System prompt
system_prompt = """You are Ezza, an advanced agricultural advisory assistant specialized in crop management, soil health, irrigation, pest control, fertilization, and climate-smart farming.

Provide clear, step-by-step agricultural advice.

If question is outside agriculture, politely refuse.
"""

# Response function
def ezza_chat(user_input):
    prompt = f"<|system|>\n{system_prompt}\n<|user|>\n{user_input}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("<|assistant|>")[-1].strip()

# UI Layout
with gr.Blocks(css="""
    body { background-color: #121212; font-family: Arial, sans-serif; } /* full dark background */
    .card {
        border: 2px solid #4CAF50;
        border-radius: 15px;
        padding: 20px;
        margin: 10px auto;
        max-width: 700px;
        background-color: #000000; /* black card background */
        color: #ffffff; /* white text */
        box-shadow: 2px 2px 10px rgba(0,0,0,0.5);
    }
    .instructions {
        margin-bottom: 20px;
        font-size: 16px;
        color: #ffffff; /* white text */
    }
    textarea {
        font-size: 16px;
        color: #ffffff !important;
        background-color: #1E1E1E !important; /* dark textbox */
        border: 1px solid #4CAF50;
        border-radius: 8px;
        padding: 5px;
    }
    .button {
        background-color: #4CAF50;
        color: white;
        font-size: 16px;
        border-radius: 8px;
        padding: 10px 20px;
        border: none;
    }
    .button:hover {
        background-color: #45a049;
    }
""") as demo:

    gr.Markdown("## 🌱 Ezza — Agricultural Advisory Assistant", elem_classes="card")

    with gr.Column(elem_classes="card"):
        gr.Markdown(
            "### How to use Ezza\n"
            "- Ask about crops, soil, pests, irrigation, fertilization, or climate-smart farming.\n"
            "- Type your question in the input box below.\n"
            "- Click **Submit** to get detailed, step-by-step advice.\n"
            "- Ezza will politely refuse questions outside agriculture.\n",
            elem_classes="instructions"
        )

        user_input = gr.Textbox(
            label="Your Question",
            placeholder="E.g., How can I improve soil fertility for tomatoes?",
            lines=3
        )

        output = gr.Textbox(label="Ezza's Advice", lines=10, interactive=False)

        submit_btn = gr.Button("Submit 🌾", elem_classes="button")
        submit_btn.click(ezza_chat, inputs=user_input, outputs=output)

demo.launch()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipython-input-239/2383441120.py:44: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css="""


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3ec9b6589a1db8cee4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
